# 03b RQ3 exploratory: CLOB USD vs. capital-at-risk denominator

**Exploratory (Section 8).** Nothing here is pre-registered, and none of it changes the
confirmatory RQ3 result in notebook 03.

**Why.** On Polymarket's CTF, buying 1 `token1` at price `p` and splitting `$1` + selling
1 `token2` at `1 - p` have identical P&L and identical capital at risk (`p`), which is why
the P&L numerator in `02_mart_wallet_signal.sql` is invariant to that choice (D6.2).
However, the CLOB `usd_amount` denominator is **not** invariant: selling a 5¢ token risks
95¢ but records only 5¢ of CLOB USD, inflating measured return-per-USD by up to 19× for
cheap-token sellers. Similarly, weighting the smart-money signal by CLOB USD underweights
cheap-token sells relative to expensive-token buys of the same share size.

`sql/03_marts/06_rq3_risk_capital.sql` builds economic **capital at risk** (`tokens * p`
for buys, `tokens * (1 - p)` for sells) and **share notional** (`tokens`) locally from
`int_wallet_trades.parquet`, and decomposes:
1. Who sits in the CLOB-USD top decile vs. the risk-capital top decile.
2. Whether the RQ3 slope and its one-sidedness (heavy selling only) come from the
   track-record denominator, the signal weighting, or persist under fully invariant
   definitions.


In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.special import expit, logit
from statsmodels.tools.sm_exceptions import ConvergenceWarning, PerfectSeparationWarning

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
CLIP = (0.001, 0.999)
pd.set_option("display.width", 200)

prof = pd.read_csv(RESULTS / "rq3_denominator_profile.csv")
prof


,grp,n_wallets,med_record_clob,med_record_risk,med_usd_to_risk,mean_sell_share,mean_cheap_sell_share,share_smart_risk,share_smart_clob
0,CLOB-USD non-smart qualifying,605519,-0.00970,-0.01013,1.0379,0.6403,0.1886,0.0225,0.0000
1,CLOB-USD top decile,67280,0.09501,0.09537,0.9997,0.6142,0.1648,0.8341,1.0000
2,Risk-capital non-smart qualifying,678515,-0.01044,-0.00349,0.9050,0.6186,0.2499,0.0000,0.0120
3,Risk-capital top decile,75391,0.08732,0.08255,1.0348,0.6241,0.1457,1.0000,0.7444


## E1. Decomposed logit refits (track-record denominator × signal weight)

Five variants on the exact 351,535 primary 1-day markets (CR1 cluster-robust SEs, 95% CIs
only):
- **original (CLOB wallets, CLOB wt):** reproduces confirmatory RQ3 (`signal`).
- **CLOB wallets, risk-cap wt:** keeps CLOB-USD smart wallets, weights their trades by
  capital at risk (isolates signal weighting).
- **risk-cap wallets, CLOB wt:** selects smart wallets by `pnl / risk_capital` (`>= 20`
  markets, `>= $1,000` risk capital), weights their trades by CLOB USD (isolates track
  record).
- **risk-cap wallets, risk-cap wt:** fully invariant to buy-YES vs. split+sell-NO.
- **risk-cap wallets, share wt:** invariant, weighted by `token_amount`.


In [2]:
base = pd.read_parquet(
    DATA / "mart_analysis.parquet",
    columns=["market_id", "horizon", "is_primary", "snap_price", "cluster_id", "token1_won", "signal", "smart_usd"],
)
d = base[(base["horizon"] == "1d") & base["is_primary"]].reset_index(drop=True)
rc = pd.read_parquet(DATA / "mart_rq3_risk_capital.parquet")
d = d.merge(rc, on="market_id", how="inner", validate="one_to_one")
assert len(d) == 351_535
assert np.max(np.abs(d["signal"] - d["signal_clob_chk"])) < 1e-9, "CLOB signal mismatch"
d["y"] = d["token1_won"].astype(float)
x = logit(d["snap_price"].clip(*CLIP).to_numpy())
groups = pd.factorize(d["cluster_id"])[0]


def fit_sig(sig_col):
    s = d[sig_col].to_numpy()
    X = np.column_stack([np.ones_like(x), x, s])
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        res = sm.Logit(d["y"].to_numpy(), X).fit(
            disp=0, maxiter=200, cov_type="cluster", cov_kwds={"groups": groups}
        )
    bad = any(issubclass(w.category, (ConvergenceWarning, PerfectSeparationWarning)) for w in caught)
    b, se = res.params, res.bse
    eta0 = X[:, :-1] @ b[:-1]
    return {
        "estimable": bool(res.mle_retvals["converged"]) and not bad,
        "n_with_smart": int((s != 0).sum()),
        "mean_signal": float(s.mean()),
        "coef": b[-1],
        "se": se[-1],
        "ci_low": b[-1] - 1.959964 * se[-1],
        "ci_high": b[-1] + 1.959964 * se[-1],
        "ame_0_to_1": float(np.mean(expit(eta0 + b[-1]) - expit(eta0))),
    }


variants = [
    ("CLOB wallets, CLOB wt (original)", "signal"),
    ("CLOB wallets, risk-cap wt", "signal_clob_w_risk"),
    ("risk-cap wallets, CLOB wt", "signal_risk_w_clob"),
    ("risk-cap wallets, risk-cap wt", "signal_risk"),
    ("risk-cap wallets, share wt", "signal_shares"),
]
decomp = pd.DataFrame([{"variant": label, "signal_col": col, **fit_sig(col)} for label, col in variants])
decomp.to_csv(RESULTS / "rq3_denominator_decomp.csv", index=False)
decomp.round(5)


,variant,signal_col,estimable,n_with_smart,mean_signal,coef,se,ci_low,ci_high,ame_0_to_1
0,"CLOB wallets, CLOB wt (original)",signal,True,272331,0.07855,0.03800,0.00943,0.01952,0.05648,0.00495
1,"CLOB wallets, risk-cap wt",signal_clob_w_risk,True,273585,-0.16502,0.06338,0.00976,0.04425,0.08251,0.00830
2,"risk-cap wallets, CLOB wt",signal_risk_w_clob,True,287061,0.13455,0.04238,0.00856,0.02560,0.05915,0.00551
3,"risk-cap wallets, risk-cap wt",signal_risk,True,288530,-0.07149,0.06953,0.00863,0.05262,0.08644,0.00910
4,"risk-cap wallets, share wt",signal_shares,True,285444,0.16283,0.03503,0.00872,0.01795,0.05211,0.00456


## E2. Does one-sidedness (heavy selling only) persist under invariant risk-capital signals?

Signal-bin breakdown of mean `(won - price)` with CR1 95% CIs for `signal_risk` and
`signal_shares`, matching notebook 03 Section 4.


In [3]:
def mean_resid(df):
    r = sm.OLS((df["y"] - df["snap_price"]).to_numpy(), np.ones(len(df))).fit(
        cov_type="cluster", cov_kwds={"groups": pd.factorize(df["cluster_id"])[0]}
    )
    b, se = r.params[0], r.bse[0]
    return {
        "markets": len(df), "clusters": df["cluster_id"].nunique(),
        "mean_price": df["snap_price"].mean(), "win_rate": df["y"].mean(),
        "resid": b, "ci_low": b - 1.959964 * se, "ci_high": b + 1.959964 * se,
    }


order = ["no smart volume", "[-1, -0.5)", "[-0.5, 0)", "[0, 0.5)", "[0.5, 1]"]
bin_tables = []
for name, sig_col, vol_col in [
    ("original (CLOB)", "signal", "smart_usd"),
    ("risk-cap invariant", "signal_risk", "smart_risk_cap"),
    ("share-weighted invariant", "signal_shares", "smart_shares"),
]:
    s = d[sig_col]
    has = d[vol_col] > 0
    sb = np.select([~has, s < -0.5, s < 0, s < 0.5], order[:-1], order[-1])
    for bname in order:
        bin_tables.append({"variant": name, "signal_bin": bname, **mean_resid(d[sb == bname])})
rc_bins = pd.DataFrame(bin_tables)
rc_bins.to_csv(RESULTS / "rq3_denominator_bins.csv", index=False)
rc_bins.round(5)


,variant,signal_bin,markets,clusters,mean_price,win_rate,resid,ci_low,ci_high
0,original (CLOB),no smart volume,78175,4406,0.38802,0.38717,-0.00085,-0.00521,0.00351
1,original (CLOB),"[-1, -0.5)",74991,7643,0.31796,0.30869,-0.00927,-0.01263,-0.00591
2,original (CLOB),"[-0.5, 0)",43828,5772,0.29296,0.29353,0.00058,-0.00303,0.00419
3,original (CLOB),"[0, 0.5)",51050,5796,0.28946,0.29475,0.00529,0.00192,0.00866
4,original (CLOB),"[0.5, 1]",103491,7572,0.24883,0.24842,-0.00041,-0.00267,0.00186
5,risk-cap invariant,no smart volume,62454,4306,0.36930,0.36899,-0.00030,-0.00507,0.00446
6,risk-cap invariant,"[-1, -0.5)",122350,9486,0.19849,0.18940,-0.00909,-0.01117,-0.00702
7,risk-cap invariant,"[-0.5, 0)",42806,4365,0.30997,0.30788,-0.00209,-0.00628,0.00210
8,risk-cap invariant,"[0, 0.5)",31766,3892,0.44852,0.46065,0.01213,0.00631,0.01796
9,risk-cap invariant,"[0.5, 1]",92159,7783,0.35461,0.35815,0.00354,0.00070,0.00639


## Summary (exploratory)

- **Track-record overlap is high (83.4%), so the wallet list shifts modestly.** At data
  end, 83.4% of CLOB-USD top-decile wallets also sit in the risk-capital top decile
  (75,391 vs. 67,280 wallets, because more wallets reach $1,000 of risk capital than
  $1,000 of CLOB USD). Switching only the wallet selection raises β slightly from
  +0.0380 to +0.0424 [+0.0256, +0.0591].
- **Signal weighting drives most of the jump (E1).** Weighting smart-wallet legs by
  economic capital at risk instead of CLOB USD raises β to +0.0634 [+0.0442, +0.0825]
  with CLOB wallets and to **+0.0695 [+0.0526, +0.0864]** (AME +0.91 pp) when both
  track records and signal weights use capital at risk. Share-notional weighting stays
  close to the original (+0.0350 [+0.0179, +0.0521]).
- **The signal becomes two-sided under risk-capital weighting (E2).** Heavy smart
  selling ([−1, −0.5)) remains −0.91¢ [−1.12, −0.70], mild buying ([0, 0.5)) rises to
  +1.21¢ [+0.63, +1.80], and heavy smart buying ([0.5, 1]) turns positive at +0.35¢
  [+0.07, +0.64] instead of −0.04¢.
- **Reading:** Part of the registered signal's one-sidedness was an artifact of CLOB-USD
  signal weighting (which gives 19× more weight to buying a 95¢ favorite than selling a
  5¢ longshot of the same share size). Even with the invariant risk-capital signal,
  however, the effect stays under 1 percentage point before spreads and fees.
